# 03 · ResNet on CIFAR-10 —— 残差连接：深度学习真正的分水岭 ★

**家族位置**：`02_CNN_Family` 第 3 个项目（家族重点）。

**与上一站的衔接**：02 项目留下 NiN 的"病"——深层网络越深越差（NiNMini 45.99%）。本章开药方，而且药方有两味：**BN**（稳定每层输入分布）+ **残差**（给梯度留直达通道）。本项目做家族第一次**五臂受控消融**：同深度、同通道，只拨动"残差"或"BN"一个开关，让每个组件的贡献各自现形。

**学习目标**
1. 理解退化问题（degradation）：深层 Plain 网络更差不是过拟合，而是"恒等映射学起来难"
2. 残差连接 $F(x)+x$ 为什么有效：恒等默认路径 + 梯度直达通道
3. 五臂受控消融：ResNet20 / Plain20 / ResNet32 / Plain32 / ResNet20-无BN
4. 量化残差的**参数代价**（约 0.1%）与精度收益的巨大反差

## 1. 原理：退化问题与残差学习

### 退化问题（Degradation）：加深反而变差

直觉是"网络越深表达力越强"。但 He 等人（2015）实测：56 层 Plain 网络的**训练误差**比 20 层还高——注意是训练误差，不是测试误差，所以**不是过拟合，是优化本身失败**：深层堆叠的网络连"什么都不做"（恒等映射）都学不好。

### 残差连接：把学习目标改成"学增量"

普通层直接学映射 $H(x)$；残差块改为学残差 $F(x) = H(x) - x$，输出变为：

$$y = F(x) + x$$

三个直接好处：
- 若某层最优解接近恒等映射，只需把 $F$ 压向 0（比学恒等容易得多）——**恒等成为默认路径**
- 反向传播时 $\dfrac{\partial L}{\partial x} = \dfrac{\partial L}{\partial y}\left(1 + \dfrac{\partial F}{\partial x}\right)$——那个 **"1"** 让梯度可以不衰减地直达浅层
- 维度不匹配处用 **1×1 投影捷径**对齐（本项目恰好 2 条，共 2,752 参数，约占 0.1%）

### 与 BN 的分工

BN 稳住每层输入的分布（02 项目已实证 batch=2 时它有多重要），残差保住梯度通道——两味药治的不是同一个病。本章用 `use_bn=False` 的残差网把它们拆开称重，兑现 02 的伏笔。

### 五臂受控消融设计

| 臂 | residual | use_bn | 深度 | 回答的问题 |
|---|---|---|---|---|
| ResNet20 | ✓ | ✓ | 20 量级 | 基准 |
| Plain20 | ✗ | ✓ | 同上 | **残差值多少**（同深度对照） |
| ResNet32 | ✓ | ✓ | 32 量级 | 深度加深，残差网受益吗 |
| Plain32 | ✗ | ✓ | 同上 | **退化问题复现**：加深惩罚 Plain |
| ResNet20-无BN | ✓ | ✗ | 20 量级 | **BN 值多少**（NiN 伏笔） |

深度按原论文 CIFAR 公式 $6n+2$：$n=3$→20 量级、$n=5$→32 量级；通道 16→32→64 与原论文一致。

In [ ]:
import sys
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import CIFAR10_CLASSES, load_cifar10_torch
from common.engine import fit
from common.models import ResNetCIFAR
from common.utils import count_params, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

## 2. 数据：CIFAR-10

10 类彩色小图（32×32×3）：飞机/汽车/鸟/猫/鹿/狗/蛙/马/船/卡车。与 Fashion-MNIST 相比多了**颜色通道 + 真实背景**——小模型在这里的绝对精度不高（SOTA ~99.5% 依赖大模型+长训），但**相对差距**正是消融要看的。训练用 10k 子集控制 CPU 预算，测试始终全量 10k。

In [ ]:
Xtr, ytr, Xte, yte = load_cifar10_torch(str(ROOT / "data"))
print("训练集:", Xtr.shape, "| 测试集:", Xte.shape)
print("类别分布(全量训练):", np.bincount(ytr.numpy()).tolist())

fig, axes = plt.subplots(2, 10, figsize=(12, 3.0))
rng = np.random.default_rng(0)
for r in range(2):
    for c in range(10):
        idx = int(np.where(ytr.numpy() == c)[0][r])
        img = Xtr[idx].permute(1, 2, 0).numpy()
        img = (img * [0.2470, 0.2435, 0.2616] + [0.4914, 0.4822, 0.4465]).clip(0, 1)
        axes[r, c].imshow(img)
        axes[r, c].set_title(CIFAR10_CLASSES[c], fontsize=8)
        axes[r, c].axis("off")
plt.suptitle("CIFAR-10：每类两个样本（反标准化后显示）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 模型定稿：一个类生成五个臂

`ResNetCIFAR` 的 `residual` / `use_bn` 两个开关就是消融本体——五个臂的代码路径逐模块一致，唯一差别是开关。先核对三笔账：
1. **参数账**：Plain 与 ResNet 同参数（残差本体不加参数），唯一差值是两条 1×1 投影捷径
2. **深度账**：卷积层数 stem + 2×块数 + 投影数（20 量级 vs 32 量级）
3. **BN 账**：无 BN 臂 BN 层数为 0，卷积 bias 自动开启补偿

In [ ]:
ARMS = {
    "ResNet20":      dict(num_blocks=(3, 3, 3), residual=True,  use_bn=True),
    "Plain20":       dict(num_blocks=(3, 3, 3), residual=False, use_bn=True),
    "ResNet32":      dict(num_blocks=(5, 5, 5), residual=True,  use_bn=True),
    "Plain32":       dict(num_blocks=(5, 5, 5), residual=False, use_bn=True),
    "ResNet20-无BN": dict(num_blocks=(3, 3, 3), residual=True,  use_bn=False),
}

for name, kw in ARMS.items():
    torch.manual_seed(0)
    m = ResNetCIFAR(**kw)
    n_conv = sum(1 for mod in m.modules() if isinstance(mod, nn.Conv2d))
    n_bn = sum(1 for mod in m.modules() if isinstance(mod, nn.BatchNorm2d))
    print(f"{name:14s} 卷积层={n_conv:2d} BN层={n_bn:2d} 参数量={count_params(m):>8,}")

p_res = count_params(ResNetCIFAR(residual=True))
p_plain = count_params(ResNetCIFAR(residual=False))
print(f"\n残差的参数代价: {p_res - p_plain:,}（占 {1 - p_plain / p_res:.2%}）——两条 1×1 投影捷径")
assert 0 < p_res - p_plain <= 3000, "残差应只引入极少量参数"

## 4. 主实验：五臂同台（约 45 分钟 CPU）

**协议（v2，经两轮原型校准）**：10k 训练子集、15 epochs、**SGD(momentum=0.9, lr=0.05) + weight_decay=1e-4**、batch 128、seed=0、测试全量 10k；不用增强（增强的收益/代价已由 02 项目专题验证，这里追求五臂严格可比）。

> **协议演进实录（本身就是教学点）**：第一版用 Adam(1e-3) 跑出反常结果——ResNet20(50.9%) 反而低于 Plain20(56.4%)，train/val 差距高达 25pt（严重过拟合）。诊断：lr=1e-3 对 BN 深网过大。ResNet 论文的标准配方是 SGD+momentum+wd（原论文用 0.1 起步 + 余弦/阶梯衰减），降到 0.05 换 v2 后排序恢复（原型：ResNet20 61.4% > Plain20 50.5%）。**教训：配方不对时，任何组件都可能在错误的实验里"被失效"。**

预期方向（跑完对答案）：残差臂显著高于同深度 Plain 臂；Plain 加深不升反降；无 BN 臂明显掉队。

In [ ]:
SUBSET, EPOCHS = 10000, 15
tr = DataLoader(TensorDataset(Xtr[:SUBSET], ytr[:SUBSET]), batch_size=128, shuffle=True)
te = DataLoader(TensorDataset(Xte, yte), batch_size=512)

results, models = {}, {}  # SGD+momentum 是 ResNet 论文原配方；Adam 在此配置下会过拟合振荡（见 §4 协议实录）
for name, kw in ARMS.items():
    set_seed(0)
    model = ResNetCIFAR(**kw)
    hist = fit(model, tr, te, epochs=EPOCHS, lr=0.05,
               optimizer_cls=partial(torch.optim.SGD, momentum=0.9),
               weight_decay=1e-4, device=DEVICE, verbose=False)
    results[name] = hist
    models[name] = model
    print(f"{name:14s} val_acc={hist['val_acc'][-1]:.2%} | val_loss={hist['val_loss'][-1]:.4f} | train_acc={hist['train_acc'][-1]:.2%}")

In [ ]:
colors = {"ResNet20": "#DD8452", "Plain20": "#4C72B0",
          "ResNet32": "#C44E52", "Plain32": "#55A868", "ResNet20-无BN": "#8172B2"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for name, c in colors.items():
    axes[0].plot(results[name]["val_acc"], marker="o", ms=4, label=name, color=c)
    axes[1].plot(results[name]["train_acc"], marker="o", ms=4, label=name, color=c)
axes[0].set_title("val_acc（CIFAR-10 全量 10k 测试）")
axes[1].set_title("train_acc（10k 子集）")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

accs = {n: results[n]["val_acc"][-1] for n in ARMS}
fig, ax = plt.subplots(figsize=(8, 4))
names = list(ARMS)
bars = ax.bar(names, [accs[n] for n in names], color=[colors[n] for n in names])
for b, n in zip(bars, names):
    ax.text(b.get_x() + b.get_width() / 2, accs[n], f"{accs[n]:.2%}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0.15, 0.75); ax.set_ylabel("val_acc")
ax.set_title("五臂受控消融（10k 子集 · 15 epochs · SGD-momentum-wd · seed=0）")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"残差增益 @20层: {accs['ResNet20'] - accs['Plain20']:+.2%} | @32层: {accs['ResNet32'] - accs['Plain32']:+.2%}")
print(f"深度效应 Plain: {accs['Plain32'] - accs['Plain20']:+.2%} | ResNet: {accs['ResNet32'] - accs['ResNet20']:+.2%}")
print(f"BN 贡献 @ResNet20: {accs['ResNet20'] - accs['ResNet20-无BN']:+.2%}")

## 5. 退化问题特写：深度是 Plain 的毒药、ResNet 的补药

把同族的 20 层与 32 层放在一起看曲线——原论文最重要的图形语言，用我们的数据复现一遍。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for name, c in [("Plain20", "#4C72B0"), ("Plain32", "#55A868")]:
    axes[0].plot(results[name]["val_acc"], marker="o", ms=4, label=name, color=c)
for name, c in [("ResNet20", "#DD8452"), ("ResNet32", "#C44E52")]:
    axes[1].plot(results[name]["val_acc"], marker="o", ms=4, label=name, color=c)
axes[0].set_title("Plain 族：加深（20→32）")
axes[1].set_title("ResNet 族：加深（20→32）")
for ax in axes:
    ax.set_xlabel("epoch"); ax.set_ylabel("val_acc"); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig3_depth.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Plain 族加深变化:   {accs['Plain32'] - accs['Plain20']:+.2%}")
print(f"ResNet 族加深变化:  {accs['ResNet32'] - accs['ResNet20']:+.2%}")
print(f"Plain 族加深后 train_acc 变化: {results['Plain32']['train_acc'][-1] - results['Plain20']['train_acc'][-1]:+.2%}（训练集上也差 → 优化失败而非过拟合）")

## 6. 冠军的错误：CIFAR-10 的混淆是"视觉语义"级的

冠军模型的混淆矩阵——猫狗相认、鹿马难分这类**视觉语义相近**的错误，和 01/02 家族"笔形/款式"错误又不一样。

In [ ]:
best_name = max(accs, key=accs.get)
print("冠军:", best_name, f"{accs[best_name]:.2%}")
best_model = models[best_name]
best_model.eval()
with torch.no_grad():
    pred = best_model(Xte).argmax(1)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte.numpy(), pred.numpy())
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(CIFAR10_CLASSES, fontsize=8)
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=6)
ax.set_xlabel("预测"); ax.set_ylabel("真实"); ax.set_title(f"{best_name} 混淆矩阵")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(FIGS / "fig4_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
flat = cm_off.ravel().argsort()[::-1][:5]
for k in flat:
    r, c = np.unravel_index(k, cm.shape)
    print(f"  {CIFAR10_CLASSES[r]:10s} → {CIFAR10_CLASSES[c]:10s} : {cm_off[r, c]} 次")

## 7. 总结与下一步

**本项目收获**

1. 残差连接的三个关键理解：恒等默认路径、梯度直达通道（公式里的那个 "1"）、1×1 投影处理维度变化
2. 五臂消融量化：残差同深度增益（实测见 §4）、BN 贡献（NiN 伏笔兑现）、深度对 Plain 的退化 vs 对 ResNet 的增益
3. 2,752 参数（0.1%）撬动的巨大收益——工程视角"性价比之王"
4. 退化问题的复现与解释：训练误差变差 = 优化失败 ≠ 过拟合

**下一步**：`04_DenseNet_Inception_CIFAR10`——继续连接拓扑的故事：DenseNet 的密集连接（每层都接所有层）vs Inception 的多尺度并行分支，与 ResNet 的相加连接同台对比。